# Assignment 06 · Notebook 00: cơ sở lý thuyết mạng hồi quy bằng NumPy

**Sinh viên:** Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · **GVHD:** PGS.TS Trần Đình Quế

Mỗi khái niệm của Chương 1 là một hàm trong `a06/theory.py`. Notebook này chạy từng hàm và đối chiếu với lớp
PyTorch tương ứng bằng `np.allclose`, rồi xuất dữ liệu cho các hình pgfplots.

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
import torch
from torch import nn
from a06 import theory as th
from a06.data_churn import load_churn
from a06.data_stock import prepare_stock
from a06.export import save_metrics, write_dat, write_matrix_dat

torch.manual_seed(0)
rng = np.random.default_rng(0)
checks = {}

## 1. Dữ liệu chuỗi là tensor $X \in \mathbb{R}^{B \times T \times D}$

Hai tập của bài đều đưa về đúng dạng này: $B$ mẫu, mỗi mẫu $T$ bước thời gian, mỗi bước $D$ đặc trưng.
Các bước liền nhau phụ thuộc nhau, nên giả định i.i.d. của MLP không còn đúng theo trục thời gian.

In [2]:
S = prepare_stock()
C = load_churn()
shapes = {"amzn": list(S["X"]["train"].shape), "kkbox": list(C["X"]["train"].shape)}
print("AMZN  train (B, T, D) =", shapes["amzn"])
print("KKBox train (B, T, D) =", shapes["kkbox"])

AMZN  train (B, T, D) = [4764, 30, 8]
KKBox train (B, T, D) = [70000, 31, 8]


## 2. Vì sao MLP và Conv1D chưa đủ: đếm tham số

Với $T = 30$, $D = 8$, $H = 64$: MLP làm phẳng chuỗi nên số tham số tăng theo $T$; Conv1D dùng chung kernel
nhưng chỉ nhìn $K$ bước; RNN dùng một bộ trọng số cho mọi bước và giữ trạng thái nhớ $h_t$.

In [3]:
T, D, H, K = 30, 8, 64, 3
params = {"mlp": th.count_params_mlp(T, D, H), "mlp_t60": th.count_params_mlp(2 * T, D, H),
          "conv1d": th.count_params_conv1d(K, D, H), "rnn": th.count_params_rnn(D, H)}
checks["params_rnn"] = params["rnn"] == sum(p.numel() for p in nn.RNN(D, H, bias=True).parameters()) - H
checks["params_conv1d"] = params["conv1d"] == sum(p.numel() for p in nn.Conv1d(D, H, K).parameters())
checks["params_mlp"] = params["mlp"] == sum(p.numel() for p in nn.Linear(T * D, H).parameters())
print(params, checks)

{'mlp': 15424, 'mlp_t60': 30784, 'conv1d': 1600, 'rnn': 4672} {'params_rnn': True, 'params_conv1d': True, 'params_mlp': True}


### Trường thụ cảm và tính nhân quả của Conv1D

Mỗi tầng Conv1D kernel $K$ chỉ nới tầm nhìn thêm $K-1$ bước: $RF = 1 + L(K-1)$. Muốn nhìn đủ 30 bước với $K=3$
phải chồng 15 tầng; tích chập giãn nở (TCN) rút xuống còn vài tầng nhưng tầm nhìn vẫn là hằng số chốt lúc thiết kế.
Conv1D `padding='same'` còn dùng cả đầu vào **sau** bước $t$, tức nhìn thấy tương lai; phải đệm lệch trái (tích
chập nhân quả) mới dùng được cho dự báo. RNN nhân quả sẵn vì $h_t$ chỉ được tính từ $x_1, \dots, x_t$.

In [4]:
rf = {"L_for_T30": next(L for L in range(1, 100) if th.receptive_field(L, K) >= T),
      "L_for_T30_dilated": next(L for L in range(1, 100) if th.receptive_field(L, K, dilated=True) >= T),
      "rf_L4": th.receptive_field(4, K), "rf_L4_dilated": th.receptive_field(4, K, dilated=True)}
leak = {"same": th.future_dependence(nn.Conv1d(D, H, K, padding="same")),
        "causal": th.future_dependence(th.CausalConv1d(D, H, K))}
xr = torch.randn(1, 10, D, requires_grad=True)
nn.RNN(D, H, batch_first=True)(xr)[0][0, 4].sum().backward()
leak["rnn"] = xr.grad[0, 5:].norm().item()
checks["causal_conv_no_leak"] = leak["causal"] == 0.0 and leak["rnn"] == 0.0 and leak["same"] > 0
print(rf, leak)

{'L_for_T30': 15, 'L_for_T30_dilated': 4, 'rf_L4': 9, 'rf_L4_dilated': 31} {'same': 2.004368782043457, 'causal': 0.0, 'rnn': 0.0}


### Bố cục tensor: batch-first và time-first

cuDNN làm việc với $(T, B, D)$ (time-first, mặc định của `nn.LSTM`); Keras và phần lớn mã PyTorch dùng
$(B, T, D)$ với `batch_first=True`. Hai bố cục chỉ là hoán vị trục, cùng trọng số cho cùng kết quả:

In [5]:
lstm_tf = nn.LSTM(D, H)                       # mặc định batch_first=False: vào (T, B, D)
lstm_bf = nn.LSTM(D, H, batch_first=True)     # vào (B, T, D)
lstm_bf.load_state_dict(lstm_tf.state_dict())
xb = torch.randn(4, T, D)
out_bf, _ = lstm_bf(xb)
out_tf, _ = lstm_tf(xb.transpose(0, 1))
checks["batch_first_equiv"] = bool(torch.allclose(out_bf, out_tf.transpose(0, 1), atol=1e-6))
layout = {"bf_in": list(xb.shape), "tf_in": list(xb.transpose(0, 1).shape)}
print(layout, checks["batch_first_equiv"])

{'bf_in': [4, 30, 8], 'tf_in': [30, 4, 8]} True


PyTorch `nn.RNN` có hai vector bias ($b_{ih}$ và $b_{hh}$) nên nhiều hơn công thức $H(D+H+1)$ đúng $H$ tham số;
phép so khớp ở trên đã trừ phần đó.

## 3. Simple RNN: $h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b)$

In [6]:
rnn = nn.RNN(D, H, batch_first=True)
x = torch.randn(1, T, D)
ref, _ = rnn(x)
p = {k: v.detach().numpy() for k, v in rnn.named_parameters()}
ours = th.rnn_forward(x[0].numpy(), p["weight_ih_l0"], p["weight_hh_l0"], p["bias_ih_l0"] + p["bias_hh_l0"])
checks["rnn_forward"] = bool(np.allclose(ours, ref[0].detach().numpy(), atol=1e-5))
print("rnn_forward khớp nn.RNN:", checks["rnn_forward"], "· shape", ours.shape)

rnn_forward khớp nn.RNN: True · shape (30, 64)


## 4. BPTT: gradient nhân qua từng bước

$\dfrac{\partial L}{\partial h_t} = \dfrac{\partial L}{\partial h_T}\prod_{k=t+1}^{T} W_{hh}^\top\,
\mathrm{diag}(1-h_k^2)$. Tích này co hoặc giãn theo hàm mũ của số bước. Đo trực tiếp bằng autograd với
$L$ chỉ phụ thuộc $h_T$. LSTM khởi tạo mặc định (bias cổng quên bằng 0, nên $f_t \approx 0{,}5$) vẫn suy giảm
gần như RNN; khi bias cổng quên dương ($b_f = 3$, $f_t \approx 0{,}95$) đường cộng của cell state mới giữ được gradient.

In [7]:
Tg = 60
g_rnn = th.grad_norms_through_time("rnn", Tg)
g_lstm = th.grad_norms_through_time("lstm", Tg)
g_lstm3 = th.grad_norms_through_time("lstm", Tg, forget_bias=3.0)
lag = np.arange(Tg)[::-1]          # số bước tính từ h_t tới h_T
write_dat("theory_gradnorm", {"lag": lag, "rnn": g_rnn / g_rnn[-1], "lstm": g_lstm / g_lstm[-1],
                              "lstm3": g_lstm3 / g_lstm3[-1]})
grad = {"T": Tg, "forget_bias": 3.0}
for name, g in (("rnn", g_rnn), ("lstm", g_lstm), ("lstm3", g_lstm3)):
    grad[f"{name}_lag20"] = g[-21] / g[-1]
    grad[f"{name}_lag59"] = g[0] / g[-1]
print({k: float(v) for k, v in grad.items()})

{'T': 60.0, 'forget_bias': 3.0, 'rnn_lag20': 4.328938538651806e-06, 'rnn_lag59': 4.0595867069168086e-17, 'lstm_lag20': 4.161906181417857e-06, 'lstm_lag59': 2.459837754465446e-14, 'lstm3_lag20': 0.01961793454434156, 'lstm3_lag59': 0.015113976038504687}


In [8]:
Tr = 40
rho = {r: th.jacobian_product_norm(r, Tr) for r in (0.9, 1.0, 1.1)}
write_dat("theory_rho", {"t": np.arange(Tr), "r09": rho[0.9], "r10": rho[1.0], "r11": rho[1.1]})
print({r: float(v[-1]) for r, v in rho.items()})

{0.9: 0.003935773747456614, 1.0: 0.2396471433233029, 1.1: 9.860228459866155}


## 5. Gradient clipping theo chuẩn

Nếu $\lVert g\rVert > c$ thì $g \leftarrow g \cdot c / \lVert g\rVert$: độ dài bị chặn, hướng giữ nguyên.

In [9]:
grads = [rng.standard_normal((H, D)) * 3, rng.standard_normal(H) * 3]
clipped, norm_before = th.clip_grad_norm(grads, 1.0)
tg = [torch.tensor(g, requires_grad=True) for g in grads]
for t_, g in zip(tg, grads):
    t_.grad = torch.tensor(g)
torch_norm = nn.utils.clip_grad_norm_(tg, 1.0)
checks["clip_grad_norm"] = bool(np.isclose(norm_before, torch_norm.item()) and
                                all(np.allclose(c, t_.grad.numpy()) for c, t_ in zip(clipped, tg)))
g2 = np.array([3.0, 4.0])
(g2c,), n2 = th.clip_grad_norm([g2], 1.0)
clip = {"norm_before": norm_before, "norm_after": float(np.sqrt(sum((c ** 2).sum() for c in clipped))),
        "g2": g2, "g2_clipped": g2c, "g2_norm": n2}
print(checks["clip_grad_norm"], clip)

True {'norm_before': np.float64(72.17723671199946), 'norm_after': 0.9999999861452166, 'g2': array([3., 4.]), 'g2_clipped': array([0.59999988, 0.79999984]), 'g2_norm': np.float64(5.0)}


## 6. Tế bào LSTM và GRU

LSTM thêm cell state $c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t$: đường cộng có cổng giúp gradient đi qua
nhiều bước. GRU gộp thành hai cổng và hoà trộn lồi $h_t = (1-z_t)\odot n_t + z_t \odot h_{t-1}$.

In [10]:
xv, hv, cv = rng.standard_normal(D), rng.standard_normal(H), rng.standard_normal(H)
lstm = nn.LSTMCell(D, H)
q = {k: v.detach().numpy() for k, v in lstm.named_parameters()}
h_ref, c_ref = lstm(torch.tensor(xv, dtype=torch.float32)[None], (torch.tensor(hv, dtype=torch.float32)[None],
                                                                  torch.tensor(cv, dtype=torch.float32)[None]))
h_new, c_new = th.lstm_cell(xv, hv, cv, q["weight_ih"], q["weight_hh"], q["bias_ih"], q["bias_hh"])
checks["lstm_cell"] = bool(np.allclose(h_new, h_ref[0].detach().numpy(), atol=1e-5) and
                           np.allclose(c_new, c_ref[0].detach().numpy(), atol=1e-5))

gru = nn.GRUCell(D, H)
q = {k: v.detach().numpy() for k, v in gru.named_parameters()}
g_ref = gru(torch.tensor(xv, dtype=torch.float32)[None], torch.tensor(hv, dtype=torch.float32)[None])
g_new = th.gru_cell(xv, hv, q["weight_ih"], q["weight_hh"], q["bias_ih"], q["bias_hh"])
checks["gru_cell"] = bool(np.allclose(g_new, g_ref[0].detach().numpy(), atol=1e-5))
cell_params = {"rnn": sum(p.numel() for p in nn.RNN(D, H).parameters()),
               "lstm": sum(p.numel() for p in nn.LSTM(D, H).parameters()),
               "gru": sum(p.numel() for p in nn.GRU(D, H).parameters()),
               "bilstm": sum(p.numel() for p in nn.LSTM(D, H, bidirectional=True).parameters())}
print(checks["lstm_cell"], checks["gru_cell"], cell_params)

True True {'rnn': 4736, 'lstm': 18944, 'gru': 14208, 'bilstm': 37888}


## 7. BiLSTM: hai chiều đọc, nối $[\overrightarrow{h}_T ; \overleftarrow{h}_1]$

In [11]:
bi = nn.LSTM(D, H, batch_first=True, bidirectional=True)
out, (hn, _) = bi(torch.randn(4, T, D))
checks["bilstm_concat"] = bool(torch.allclose(out[:, -1, :H], hn[0]) and torch.allclose(out[:, 0, H:], hn[1]))
bilstm = {"out_shape": list(out.shape), "hn_shape": list(hn.shape)}
print(bilstm, checks["bilstm_concat"])

{'out_shape': [4, 30, 128], 'hn_shape': [2, 4, 64]} True


## 8. Điều chuẩn: recurrent dropout và LayerNorm

Dropout thường đổi mặt nạ ở mỗi bước, làm nhiễu trạng thái nhớ theo thời gian; recurrent dropout giữ một mặt nạ
cho cả chuỗi. LayerNorm chuẩn hoá theo đặc trưng của từng mẫu nên không phụ thuộc kích thước lô.

In [12]:
std_mask, rec_mask = th.recurrent_dropout_masks(12, 8, 0.3)
write_matrix_dat("theory_mask_std", std_mask.T)
write_matrix_dat("theory_mask_rec", rec_mask.T)
xl = rng.standard_normal((5, H))
ln = nn.LayerNorm(H)
checks["layer_norm"] = bool(np.allclose(th.layer_norm(xl, ln.weight.detach().numpy(), ln.bias.detach().numpy()),
                                        ln(torch.tensor(xl, dtype=torch.float32)).detach().numpy(), atol=1e-5))
print(checks)
assert all(checks.values()), "có phép đối chiếu sai"

{'params_rnn': True, 'params_conv1d': True, 'params_mlp': True, 'causal_conv_no_leak': True, 'batch_first_equiv': True, 'rnn_forward': True, 'clip_grad_norm': True, 'lstm_cell': True, 'gru_cell': True, 'bilstm_concat': True, 'layer_norm': True}


In [13]:
save_metrics("theory", {"shapes": shapes, "params": params, "cell_params": cell_params, "grad": grad,
                        "rho_end": {"r09": rho[0.9][-1], "r10": rho[1.0][-1], "r11": rho[1.1][-1], "T": Tr},
                        "clip": clip, "bilstm": bilstm, "checks": checks, "n_checks": len(checks),
                        "mask_p": 0.3, "mask_keep_std": std_mask.mean(), "mask_keep_rec": rec_mask.mean(),
                        "rf": rf, "leak": leak, "layout": layout,
                        "gru_lstm_ratio": cell_params["gru"] / cell_params["lstm"]})
print("đã ghi outputs/metrics/theory.json ·", len(checks), "phép đối chiếu đều đúng")

đã ghi outputs/metrics/theory.json · 11 phép đối chiếu đều đúng
